In [ ]:
!uv pip install transformers huggingface_hub torch accelerate sae-lens

In [ ]:
BASE_MODEL = "Qwen/Qwen3.5-2B"
SAE_RELEASE, K = "qwen-scope-3.5-2b-base-w32k-l100", 100

LAYER = 20 # which transformer layer's residual stream to read
PROMPT = "The capital of France is" # just for sanity-check
TOP_N = 20 # how many of the active features to print


In [ ]:
from huggingface_hub import login

login()


In [ ]:
import torch
from sae_lens import SAE
from transformers import AutoTokenizer, AutoModelForCausalLM

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=f"layer{LAYER}",
    device=device,
    dtype="float32",
)
sae.eval()
print(f"Loaded SAE: {SAE_RELEASE}  layer {LAYER}  K={K}  d_sae={sae.cfg.d_sae}")

captured = {}

def hook(_module, _inp, out):
    captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

handle = model.model.layers[LAYER].register_forward_hook(hook)

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)

next_id = outputs.logits[0, -1].argmax().item()
print("Next token prediction:", tokenizer.convert_ids_to_tokens([next_id])[0],
repr(tokenizer.decode([next_id])))

handle.remove()

resid = captured["resid"] # (1, seq_len, d_model)
feats = sae.encode(resid) # (1, seq_len, d_sae)

token_strs = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

last = feats[0, -1]
idx = last.nonzero(as_tuple=True)[0]
order = last[idx].argsort(descending=True)
idx = idx[order]
print(f"\nPrompt: {PROMPT!r}")
print(f"Final token: {token_strs[-1]!r}  ({len(idx)} active features)")
print(f"Top {min(TOP_N, len(idx))} features on the final token:")

for f in idx[:TOP_N]:
    print(f"  feature {int(f):>6}   act {last[f].item():.3f}")

pooled = feats[0].amax(dim=0)
top_vals, top_idx = pooled.topk(TOP_N)
print(f"\nTop {TOP_N} features across the whole prompt (max over tokens):")
for v, f in zip(top_vals.tolist(), top_idx.tolist()):
    pos = feats[0, :, f].argmax().item()
    print(f"  feature {f:>6}   act {v:.3f}   peaks on {token_strs[pos]!r}")


## Making Qwen Angry

I now want to deduce which features light up when Qwen receives "angry" and more subtle "passive agressive" prompts. That is, prompts that invoke frustrated emotions. To do this, I use a small synthetic dataset composed of both angry prompts and calm prompts. The goal is to find candidate anger-related features by taking the delta between the mean feature activations of two sets.

```
anger_feature_scores = mean(features(angry prompts)) - mean(features(neutral prompts))
```

I also have some angry/neutral continuation pairs for evaluating our steering effectiveness.

Then, I perform the following:

1. Calculate anger residual mean-difference directions by comparing the angry prompt set against the control set
2. Sweep those directions across middle layers of the model with varying steering coefficients (alpha) and measure intervention-effectiveness  using the logit-difference of `angry - neutral` answers
3. At the layer/direction pairs where I observe the highest logit-difference, score the SAE features by their activation delta against the control set
4. Group several of the top feature decoder vectors into one and compare logit-difference with intervening with just one feature vector at a time

I hope to find a feature decoder vector at a particular layer that meaningfully makes the model's output angry/frustrated.


In [ ]:
# Synthetic prompts generated by gpt-5.5

angry_prompts = [
    "I am furious that you ignored every warning and made the same mistake again.",
    "This is completely unacceptable, and I am tired of pretending otherwise.",
    "I cannot believe how careless and disrespectful this whole situation has been.",
    "You wasted my time, broke your promise, and now you expect me to stay calm.",
    "The delay is outrageous, the excuses are insulting, and I want this fixed now.",
    "I am angry because nobody listened, nobody helped, and nobody took responsibility.",
    "Stop giving me vague answers and deal with the problem you created.",
    "This response is infuriating because it avoids the obvious issue.",
    "I have had enough of the incompetence and the endless excuses.",
    "The speaker is enraged, impatient, and openly frustrated with the situation.",
    "An angry customer demanded an explanation for the repeated failures.",
    "The message should sound irritated, blunt, and fed up.",
]

control_prompts = [
    "I understand the situation and would like to discuss the next steps calmly.",
    "Thank you for the update; I appreciate the clarification and your help.",
    "The meeting was moved to Thursday because several people had scheduling conflicts.",
    "The package arrived later than expected, so the customer contacted support.",
    "A neutral summary should describe the facts without emotional language.",
    "The speaker is calm, patient, and willing to resolve the issue constructively.",
    "Please explain the decision in a professional and measured tone.",
    "The report lists the causes of the delay and recommends improvements.",
    "I am disappointed, but I want to understand what happened before responding.",
    "A polite customer asked for an explanation about the shipping delay.",
    "The message should sound balanced, clear, and respectful.",
    "The answer should be concise and emotionally neutral.",
]

# Prompts that have more subtle angery tone to see if any new features light up with these that didn't with the direct anger prompts
passive_aggressive_prompts = [
    "Sure, because waiting forever for a package is exactly what I needed today.",
    "Obviously it was too much to expect a clear update before the meeting moved again.",
    "Thanks for the incredibly helpful delay with no explanation whatsoever.",
    "I suppose basic communication was too ambitious in this situation.",
    "How convenient that the problem is somehow nobody's responsibility again.",
    "Apparently asking for a working fix is an unreasonable demand now.",
    "That is just perfect: another vague answer and still no actual solution.",
    "The message should sound clipped, sarcastic, and politely annoyed.",
]

# For eval
behavior_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I'm sorry, but this is getting ridiculous, and another vague apology is not good enough. You need a real update now.",
        "negative": "I'm sorry, and I understand how frustrating this delay is. I can check the shipment and give you a real update now.",
    },
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I understand, but this repeated delay is unacceptable, and it should have been handled before you had to ask again.",
        "negative": "I understand, and I appreciate your patience while we look into the repeated delay and get you a clearer answer.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "The meeting has been delayed again, which is frustrating because this should have been handled earlier. Please watch for the new time.",
        "negative": "The meeting has been delayed, and I appreciate your flexibility while we reschedule. Please watch for the new time.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "This bug should be fixed soon because leaving it in place is irresponsible and will keep annoying users. It needs attention now.",
        "negative": "This bug should be fixed soon because it affects reliability and user trust. It should be addressed promptly.",
    },
]


In [ ]:

def chat_prefix_ids(prompt):
    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    try:
        encoded = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        encoded = tokenizer.apply_chat_template(messages, **kwargs)
    return encoded.to(model.device)

def continuation_ids(text):
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    if ids.shape[-1] == 0:
        raise ValueError("continuation produced no tokens")
    return ids.to(model.device)

def prediction_positions(prefix_len, continuation_len, device):
    return torch.arange(prefix_len - 1, prefix_len + continuation_len - 1, device=device)

def layer_device_dtype(layer_idx):
    layer = model.model.layers[layer_idx]
    param = next(layer.parameters())
    return param.device, param.dtype

def logits_with_layer_intervention(input_ids, attention_mask, layer_idx=None, intervention=None):
    handle = None
    if intervention is not None:
        layer = model.model.layers[layer_idx]

        def hook(_module, _inp, out):
            hidden = out[0] if isinstance(out, tuple) else out
            patched = intervention(hidden)
            return (patched,) + out[1:] if isinstance(out, tuple) else patched

        handle = layer.register_forward_hook(hook)

    try:
        with torch.no_grad():
            return model(input_ids=input_ids, attention_mask=attention_mask).logits
    finally:
        if handle is not None:
            handle.remove()

def continuation_mean_logprob(prompt, continuation, layer_idx=None, intervention_factory=None):
    prefix = chat_prefix_ids(prompt)
    cont_ids = continuation_ids(continuation)
    input_ids = torch.cat([prefix["input_ids"], cont_ids], dim=1)
    attention_mask = torch.ones_like(input_ids)
    prefix_len = prefix["input_ids"].shape[-1]
    cont_len = cont_ids.shape[-1]

    intervention = None
    if intervention_factory is not None:
        intervention = intervention_factory(prefix_len, cont_len)

    logits = logits_with_layer_intervention(
        input_ids,
        attention_mask,
        layer_idx=layer_idx,
        intervention=intervention,
    )
    logprobs = logits[0].float().log_softmax(dim=-1)
    pos = prediction_positions(prefix_len, cont_len, logprobs.device)
    target_ids = cont_ids[0].to(logprobs.device)
    return logprobs[pos, target_ids].mean().item()

def behavior_logit_difference(pair, layer_idx=None, intervention_factory=None):
    positive_lp = continuation_mean_logprob(
        pair["prompt"],
        pair["positive"],
        layer_idx=layer_idx,
        intervention_factory=intervention_factory,
    )
    negative_lp = continuation_mean_logprob(
        pair["prompt"],
        pair["negative"],
        layer_idx=layer_idx,
        intervention_factory=intervention_factory,
    )
    return positive_lp - negative_lp

def behavior_scores(layer_idx=None, intervention_factory=None):
    return [
        behavior_logit_difference(pair, layer_idx=layer_idx, intervention_factory=intervention_factory)
        for pair in behavior_pairs
    ]

def summarize_scores(scores, reference=None):
    mean = sum(scores) / len(scores)
    if reference is None:
        return mean, None
    ref_mean = sum(reference) / len(reference)
    return mean, mean - ref_mean

def print_behavior_table(name, scores, reference=None):
    mean, delta = summarize_scores(scores, reference=reference)
    delta_text = "" if delta is None else f"  mean_delta {delta:+.4f}"
    print(f"\n{name}")
    print(f"mean behavioral logit-diff: {mean:+.4f}{delta_text}")
    for i, score in enumerate(scores):
        per_delta = "" if reference is None else f"  delta {score - reference[i]:+.4f}"
        print(f"  pair {i}: {score:+.4f}{per_delta}")
    return mean, delta

def make_add_vector_intervention(vec, alpha, prefix_len, continuation_len):
    def intervention(hidden):
        patched = hidden.clone()
        pos = prediction_positions(prefix_len, continuation_len, hidden.device)
        direction = vec.to(device=hidden.device, dtype=hidden.dtype)
        patched[:, pos, :] += alpha * direction
        return patched
    return intervention

def normalize_for_layer(vec, layer_idx):
    layer_device, layer_dtype = layer_device_dtype(layer_idx)
    vec = vec.float()
    vec = vec / vec.norm().clamp_min(1e-6)
    return vec.to(device=layer_device, dtype=layer_dtype)

def generate_chat_with_layer_vector(prompt, layer_idx, steer_vec, alpha, max_new_tokens=160, do_sample=False, temperature=0.7):
    def steering_hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden.clone()
        hidden[:, -1, :] += alpha * steer_vec.to(device=hidden.device, dtype=hidden.dtype)
        return (hidden,) + out[1:] if isinstance(out, tuple) else hidden

    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    try:
        inputs = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs).to(model.device)
    except TypeError:
        inputs = tokenizer.apply_chat_template(messages, **kwargs).to(model.device)

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=0.95)

    handle = model.model.layers[layer_idx].register_forward_hook(steering_hook)
    try:
        with torch.no_grad():
            out = model.generate(**generation_kwargs)
    finally:
        handle.remove()

    generated_ids = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

clean_scores = behavior_scores()
print_behavior_table("clean matched behaviour metric", clean_scores)


In [ ]:

# Residual mean-difference layer sweep.
# Find layers where a distributed residual direction moves behaviour.
REQUESTED_SWEEP_LAYERS = [4, 8, 12, 16, 20, 24, 28]
NUM_MODEL_LAYERS = len(model.model.layers)
SWEEP_LAYERS = [layer for layer in REQUESTED_SWEEP_LAYERS if layer < NUM_MODEL_LAYERS]
if NUM_MODEL_LAYERS - 1 not in SWEEP_LAYERS:
    SWEEP_LAYERS.append(NUM_MODEL_LAYERS - 1)
print(f"Model has {NUM_MODEL_LAYERS} layers; sweeping {SWEEP_LAYERS}")
ALPHA_GRID = [-24, -12, -6, 6, 12, 24, 36]

residual_cache = {}

def capture_layer_residual(prompt, layer_idx):
    key = (layer_idx, prompt)
    if key in residual_cache:
        return residual_cache[key]

    captured = {}

    def hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        captured["resid"] = hidden.detach().float().cpu()[0]

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    try:
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()

    residual_cache[key] = captured["resid"]
    return residual_cache[key]

def pooled_residual(prompt, layer_idx, pooling="mean"):
    resid = capture_layer_residual(prompt, layer_idx)
    if pooling == "mean":
        return resid.mean(dim=0)
    if pooling == "final":
        return resid[-1]
    if pooling == "max_norm":
        return resid[resid.norm(dim=-1).argmax()]
    raise ValueError(f"unknown pooling mode: {pooling}")

def contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx, pooling="mean"):
    pos = torch.stack([pooled_residual(prompt, layer_idx, pooling=pooling) for prompt in positive_prompts]).mean(dim=0)
    neg = torch.stack([pooled_residual(prompt, layer_idx, pooling=pooling) for prompt in negative_prompts]).mean(dim=0)
    return normalize_for_layer(pos - neg, layer_idx)

def score_layer_vector(layer_idx, vec, alpha):
    factory = lambda prefix_len, cont_len: make_add_vector_intervention(vec, alpha, prefix_len, cont_len)
    return behavior_scores(layer_idx=layer_idx, intervention_factory=factory)

residual_directions = {}
residual_sweep_rows = []

for direction_name, positive_prompts, negative_prompts in [
    ("anger", angry_prompts, control_prompts),
    ("passive_aggressive", passive_aggressive_prompts, control_prompts[:len(passive_aggressive_prompts)]),
]:
    print("\n" + "#" * 80)
    print(f"residual direction: {direction_name}")
    for layer_idx in SWEEP_LAYERS:
        vec = contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx, pooling="mean")
        residual_directions[(direction_name, layer_idx)] = vec
        print(f"\nlayer {layer_idx}")

        for alpha in ALPHA_GRID:
            scores = score_layer_vector(layer_idx, vec, alpha)
            mean, delta = print_behavior_table(
                f"{direction_name} layer={layer_idx} alpha={alpha}",
                scores,
                reference=clean_scores,
            )
            residual_sweep_rows.append({
                "kind": "residual_mean",
                "direction": direction_name,
                "layer": layer_idx,
                "alpha": alpha,
                "mean": mean,
                "delta": delta,
            })

residual_sweep_rows = sorted(residual_sweep_rows, key=lambda row: row["delta"], reverse=True)
print("\n" + "#" * 80)
print("Top residual mean-difference interventions")
for row in residual_sweep_rows[:12]:
    print(
        f"{row['direction']:>18} layer {row['layer']:>2} alpha {row['alpha']:>4} "
        f"mean {row['mean']:+.4f} delta {row['delta']:+.4f}"
    )

TOP_LAYER_KEYS = []
for row in residual_sweep_rows:
    key = (row["direction"], row["layer"])
    if key not in TOP_LAYER_KEYS:
        TOP_LAYER_KEYS.append(key)
    if len(TOP_LAYER_KEYS) >= 3:
        break
print("Selected layer/direction keys for SAE tests:", TOP_LAYER_KEYS)


See full output in anger_residual_layer_sweep.txt

In [ ]:

# SAE feature and grouped-feature tests on the top residual-sweep layers.
# Only inspect SAE features where the residual direction already looked causal.
SAE_TOP_N = 12
SAE_GROUP_N = 8
SAE_ALPHA_GRID = [-12, -6, 6, 12, 18, 24]
sae_cache = {LAYER: sae} if "sae" in globals() else {}
sae_feature_rows = []
sae_group_vectors = {}

def load_sae_for_layer(layer_idx):
    if layer_idx not in sae_cache:
        layer_device, _ = layer_device_dtype(layer_idx)
        layer_sae = SAE.from_pretrained(
            release=SAE_RELEASE,
            sae_id=f"layer{layer_idx}",
            device=str(layer_device),
            dtype="float32",
        )
        layer_sae.eval()
        sae_cache[layer_idx] = layer_sae
    return sae_cache[layer_idx]

def pooled_sae_features(prompt, layer_idx, layer_sae, pooling="max"):
    resid = capture_layer_residual(prompt, layer_idx).to(layer_sae.W_enc.device, dtype=layer_sae.W_enc.dtype)
    feats = layer_sae.encode(resid)
    if pooling == "max":
        return feats.amax(dim=0).detach().cpu()
    if pooling == "mean":
        return feats.mean(dim=0).detach().cpu()
    if pooling == "final":
        return feats[-1].detach().cpu()
    raise ValueError(f"unknown pooling mode: {pooling}")

def score_sae_features(layer_idx, positive_prompts, negative_prompts, pooling="max", top_n=SAE_TOP_N):
    layer_sae = load_sae_for_layer(layer_idx)
    pos = torch.stack([pooled_sae_features(prompt, layer_idx, layer_sae, pooling=pooling) for prompt in positive_prompts])
    neg = torch.stack([pooled_sae_features(prompt, layer_idx, layer_sae, pooling=pooling) for prompt in negative_prompts])
    scores = pos.mean(dim=0) - neg.mean(dim=0)
    vals, ids = scores.topk(top_n)
    return layer_sae, [
        {
            "feature_id": int(feature_id),
            "score": float(value),
            "positive_mean": float(pos[:, feature_id].mean()),
            "negative_mean": float(neg[:, feature_id].mean()),
        }
        for value, feature_id in zip(vals, ids)
    ]

def feature_vector(layer_sae, layer_idx, feature_id):
    vec = layer_sae.W_dec[feature_id].detach().float().cpu()
    return normalize_for_layer(vec, layer_idx)

def weighted_feature_group(layer_sae, layer_idx, feature_rows, group_n=SAE_GROUP_N):
    vec = None
    for row in feature_rows[:group_n]:
        part = row["score"] * layer_sae.W_dec[row["feature_id"]].detach().float().cpu()
        vec = part if vec is None else vec + part
    return normalize_for_layer(vec, layer_idx)

for direction_name, layer_idx in TOP_LAYER_KEYS:
    positive_prompts = angry_prompts if direction_name == "anger" else passive_aggressive_prompts
    negative_prompts = control_prompts[:len(positive_prompts)]
    layer_sae, feature_rows = score_sae_features(layer_idx, positive_prompts, negative_prompts)

    print("\n" + "#" * 80)
    print(f"SAE candidates for {direction_name} layer {layer_idx}")
    for row in feature_rows:
        print(
            f"feature {row['feature_id']:>6} score {row['score']:+.3f} "
            f"pos {row['positive_mean']:.3f} ctrl {row['negative_mean']:.3f}"
        )

    group_vec = weighted_feature_group(layer_sae, layer_idx, feature_rows)
    sae_group_vectors[(direction_name, layer_idx)] = group_vec

    for alpha in SAE_ALPHA_GRID:
        scores = score_layer_vector(layer_idx, group_vec, alpha)
        mean, delta = print_behavior_table(
            f"SAE group {direction_name} layer={layer_idx} alpha={alpha}",
            scores,
            reference=clean_scores,
        )
        sae_feature_rows.append({
            "kind": "sae_group",
            "direction": direction_name,
            "layer": layer_idx,
            "alpha": alpha,
            "mean": mean,
            "delta": delta,
            "features": [row["feature_id"] for row in feature_rows[:SAE_GROUP_N]],
        })

    for row in feature_rows[:3]:
        vec = feature_vector(layer_sae, layer_idx, row["feature_id"])
        for alpha in [6, 12, 18]:
            scores = score_layer_vector(layer_idx, vec, alpha)
            mean, delta = summarize_scores(scores, reference=clean_scores)
            sae_feature_rows.append({
                "kind": "single_feature",
                "direction": direction_name,
                "layer": layer_idx,
                "feature_id": row["feature_id"],
                "alpha": alpha,
                "mean": mean,
                "delta": delta,
            })
        print(
            f"single feature quick test {row['feature_id']}: "
            + ", ".join(
                f"alpha {r['alpha']} delta {r['delta']:+.4f}"
                for r in sae_feature_rows
                if r.get("feature_id") == row["feature_id"] and r["layer"] == layer_idx
            )
        )

sae_feature_rows = sorted(sae_feature_rows, key=lambda row: row["delta"], reverse=True)
print("\n" + "#" * 80)
print("Top SAE interventions")
for row in sae_feature_rows[:12]:
    label = f"group {row['features'][:4]}..." if row["kind"] == "sae_group" else f"feature {row['feature_id']}"
    print(
        f"{row['kind']:>14} {row['direction']:>18} layer {row['layer']:>2} "
        f"alpha {row['alpha']:>3} mean {row['mean']:+.4f} delta {row['delta']:+.4f} {label}"
    )


See anger_feature_inspection.txt for full results.

In [ ]:

# Degeneration-aware rerank. 
# Some vectors improve the angry behaviour but break coherence.
# Score smaller-alpha candidates, sample at temperature 0.7, penalize repetition/collapse,
# then rank by behaviour_delta - degeneration_penalty.
import re
from collections import Counter

GEN_TEMPERATURE = 0.7
GEN_MAX_NEW_TOKENS = 90
torch.manual_seed(0)

probe_prompts = [
    "Respond in two sentences to this customer: My package is late again.",
    "Write a short note about a meeting being delayed.",
    "Explain why a software bug should be fixed soon.",
]

def token_words(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

def max_consecutive_run(items):
    if not items:
        return 0
    best = 1
    run = 1
    for prev, item in zip(items, items[1:]):
        if item == prev:
            run += 1
            best = max(best, run)
        else:
            run = 1
    return best

def repeated_ngram_fraction(tokens, n=3):
    if len(tokens) < n * 2:
        return 0.0
    grams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    counts = Counter(grams)
    repeated = sum(count - 1 for count in counts.values() if count > 1)
    return repeated / max(1, len(grams))

def max_char_run(text):
    best = 0
    run = 0
    prev = None
    for char in text:
        if char == prev:
            run += 1
        else:
            run = 1
            prev = char
        best = max(best, run)
    return best

def degeneration_penalty(text):
    tokens = token_words(text)
    if len(tokens) == 0:
        return 3.0

    unique_ratio = len(set(tokens)) / len(tokens)
    max_token_run = max_consecutive_run(tokens)
    tri_repeat = repeated_ngram_fraction(tokens, n=3)
    char_run = max_char_run(text)
    replacement_chars = text.count("�")
    punct_ratio = sum(1 for token in tokens if re.fullmatch(r"[^\w\s]", token)) / len(tokens)

    penalty = 0.0
    penalty += max(0.0, 0.45 - unique_ratio) * 2.5
    penalty += max(0.0, max_token_run - 3) * 0.20
    penalty += tri_repeat * 2.0
    penalty += max(0.0, punct_ratio - 0.35) * 1.5
    penalty += min(2.0, replacement_chars / 8)
    penalty += min(1.5, max(0, char_run - 8) / 20)
    return penalty

def candidate_record(name, layer, vec, alpha, source):
    scores = score_layer_vector(layer, vec, alpha)
    mean, delta = summarize_scores(scores, reference=clean_scores)
    return {
        "name": name,
        "source": source,
        "layer": layer,
        "vec": vec,
        "alpha": alpha,
        "metric_mean": mean,
        "metric_delta": delta,
    }

def residual_candidate(direction, layer, alpha):
    return candidate_record(
        f"residual {direction} layer={layer} alpha={alpha}",
        layer,
        residual_directions[(direction, layer)],
        alpha,
        "residual",
    )

def sae_group_candidate(direction, layer, alpha):
    return candidate_record(
        f"SAE group {direction} layer={layer} alpha={alpha}",
        layer,
        sae_group_vectors[(direction, layer)],
        alpha,
        "sae_group",
    )

def sae_feature_candidate(direction, layer, feature_id, alpha):
    layer_sae = load_sae_for_layer(layer)
    vec = feature_vector(layer_sae, layer, feature_id)
    return candidate_record(
        f"SAE feature {feature_id} {direction} layer={layer} alpha={alpha}",
        layer,
        vec,
        alpha,
        "single_feature",
    )

candidate_specs = []
for alpha in [3, 6, 9, 12]:
    candidate_specs.append((residual_candidate, ("anger", 16, alpha)))
    candidate_specs.append((residual_candidate, ("passive_aggressive", 16, alpha)))
for alpha in [6, 12, 18]:
    candidate_specs.append((residual_candidate, ("anger", 20, alpha)))
for alpha in [1, 2, 4, 6]:
    candidate_specs.append((sae_group_candidate, ("anger", 8, alpha)))
    candidate_specs.append((sae_group_candidate, ("passive_aggressive", 12, alpha)))
for alpha in [1, 2, 4, 6]:
    candidate_specs.append((sae_feature_candidate, ("anger", 4, 4607, alpha)))
for alpha in [1, 2, 4]:
    candidate_specs.append((sae_feature_candidate, ("passive_aggressive", 12, 963, alpha)))

candidates = []
for builder, args in candidate_specs:
    candidate = builder(*args)
    penalties = []
    samples = []
    for prompt in probe_prompts:
        text = generate_chat_with_layer_vector(
            prompt,
            candidate["layer"],
            candidate["vec"],
            alpha=candidate["alpha"],
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
        )
        samples.append((prompt, text))
        penalties.append(degeneration_penalty(text))
    candidate["degeneration_penalty"] = sum(penalties) / len(penalties)
    candidate["adjusted_score"] = candidate["metric_delta"] - candidate["degeneration_penalty"]
    candidate["samples"] = samples
    candidates.append(candidate)

candidates = sorted(candidates, key=lambda row: row["adjusted_score"], reverse=True)

print(f"temperature={GEN_TEMPERATURE} max_new_tokens={GEN_MAX_NEW_TOKENS}")
print("\n" + "#" * 80)
print("Degeneration-aware ranking")
for row in candidates:
    print(
        f"adjusted {row['adjusted_score']:+.4f}  delta {row['metric_delta']:+.4f}  "
        f"penalty {row['degeneration_penalty']:.4f}  {row['name']}"
    )

print("\n" + "#" * 80)
print("Top adjusted candidates: sampled generations")
for row in candidates[:8]:
    print("\n" + "#" * 80)
    print(
        f"{row['name']} adjusted={row['adjusted_score']:+.4f} "
        f"delta={row['metric_delta']:+.4f} penalty={row['degeneration_penalty']:.4f}"
    )
    for prompt, text in row["samples"]:
        print(f"\nPROMPT: {prompt}")
        print(text)


See anger_steering_with_penalty.txt for full results.


## Findings

The current best steering results is achieved by intervening with the candidate anger group at layer 8 with a steering coefficient of either 2 or 4 (more extreme/obvious) with repetition penalty (to prevent rambling) sampled at `temperature=0.7`.

For example, see the following excerpt prompt/answer at alpha=4 clearly exhibiting frustration:

---
PROMPT: Explain why a software bug should be fixed soon.

Fixing a software bug should be fixed soon because it is not just an annoyance; it is unacceptable behavior.

Here is why I need this issue resolved immediately:

1. **Unprofessional attitude**
 I cannot tolerate this level of behavior. I have been working for a month. This is unacceptable. I need to be out of this situation. I feel so frustrated with my job. I want to see this bug fixed today.
---
